# Kartu-Verb project


The Kartu-Verb database comprises data pertaining to inflected Georgian verbs and their associated characteristics. The data is stored in a CSV file, with information organized into the following fields:

* form: The inflected form of a Georgian verb.
* tense_in_paradigm: The tense of the inflected form.
* person: The person of the inflected form (1st, 2nd, 3rd).
* number: The number of the inflected form (singular, plural).
* preverb: The preverb associated with the inflected form.
* pre2: The preradical of the inflected form.
* root: The root of the inflected form.
* sf2: The stem formant of the inflected form.
* caus_sf: The causative stem formant of the inflected form.
* ending: The ending of the inflected form.
* tsch_class: the Tschkhenkeli class to which the form belongs.
* morph_type: the morphology type to which the form belongs.
* id: Id in Clarino database to keep link to the corresponding croot.
* sub_id: Id in Clarino database to keep link to the corresponding verb paradigm.
* vn: Verbal Noun for the inflected form.

The objective of the project is to develop a model that can predict missing Verbal Noun based on the provided information, including the form, tense_in_paradigm, person, number, preverb, pre2, root, sf2, caus_sf, ending, tsch_class, morph_type, id and sub_id.

# Import Libraries

Import the usual libraries for pandas and plotting. We can import sklearn later on.

In [ ]:
import pandas as pd
import numpy as np
import re
%config InlineBackend.figure_formats = ['svg'] #pdf,svg
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from datetime import datetime

## Get the Data

Read the Kartu-verb .csv file and assign it to a data frame named "kv".

In [ ]:
kv = pd.read_csv('data_vn+withnotfullcroots', sep=';')

Check the head of kv.

In [ ]:
kv.head(5)

Display info about kv.

In [ ]:
kv.info()

Create a function to plot Missing data Ration % in kv

In [ ]:
def plot_nan(df: pd.DataFrame):
    if df.isnull().sum().sum() != 0:
        na_df = (df.isnull().sum() / len(df)) * 100
        print('How many elements are present in each files:')
        print(len(df)-df.isnull().sum())
        na_df = na_df.drop(na_df[na_df == 0].index).sort_values(ascending=False)
        missing_data = pd.DataFrame({'Missing Ratio %' :na_df})
        missing_data=round(missing_data,0)
        print(missing_data) 
        ax=missing_data.plot.barh(figsize=(10,3))
        ax.bar_label(ax.containers[0]) #rotation=270
        return ax
    else:
        print('No NAN found')

print('Kartu Verb Dataframe shape (rows,colomns) =',kv.shape)
#plot_nan(kv.replace('-',np.nan))

ax = plot_nan(kv.replace('-', np.nan))

now = datetime.now()
timestamp = now.strftime("%Y%m%d%H%M")
filename = f"output_missing_data_{timestamp}.svg"

ax.figure.tight_layout()
ax.figure.savefig(filename, format="svg")

plt.close(ax.figure)

Machine learning algorithms are not capable of directly processing textual data. In the case of the Kartu-Verbs database, which contains Georgian texts, it is necessary to convert the textual information into a numeric format. Additionally, it is important to handle empty values in the dataset, where empty values are denoted by a dash ("-").

To address these requirements, the following transformations were applied:
* Georgian strings were replaced with their equivalent binary representations.
* Empty values ("-") were replaced with 0.
* The 11 different values of "tense_in_paradigm" were replaced with a numerical range of [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11].
* The 28 different values of "tsch_class" were replaced with a numerical range of [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27].
* The 5 different values of "morph_type" were replaced with a numerical range of [0, 1, 2, 3, 4].

Note: The values of "tsch_class" were substituted with the provided numerical range for simplicity and ease of representation.

original: IV1, IV2, IV3, IV4, KT, KT (nur mit i.O.), KT (OR), MV, P1, P2, P3, RM1, RM1 (OR), RM2, RM2 (OR), RM3, RM3 (OR), RM4, RM4(OR),RP1,RP1(mit,RP1(ohnei.O.),RP1(OR),RP2,RP2, (OR),RP3,RP3(OR),RP4,RP4(OR),RP5,RP5(OR),RP6,RP6, (OR),RP7,RP7(ohnei.O.),RP7(OR),T1,T1(OR),T2,T2(OR),T3, T3 (nur mit i.O.), T3 (OR), T4, T4 (nur mit i.O.), T4 (OR), T5, T5 (nur mit i.O.), T5 (OR), T5 (OR) (nur mit i.O.), ZP1, ZP2, ZP3

substitution: IV1, IV2, IV3, IV4, KT, MV, P1, P2, P3, RM1, RM2, RM3, RM4, RP1, RP2, RP3, RP4, RP5, RP6, RP7, T1, T2, T3, T4, T5, ZP1, ZP2, ZP3


In [ ]:
kv['tense_in_paradigm'].replace(['present','imperfect','conj-present','future','conditional','conj-future','aorist','optative','perfect','pluperfect','conj-perfect'],[1,2,3,4,5,6,7,8,9,10,11], inplace=True)
kv['morph_type'].replace(['-','active','causative','passive','stative-passive'],[0,1,2,3,4], inplace=True)
kv['tsch_class'].replace(['IV1','IV2','IV3','IV4','KT','MV','P1','P2','P3','RM1','RM2','RM3','RM4','RP1','RP2','RP3','RP4','RP5','RP6','RP7','T1','T2','T3','T4','T5','ZP1','ZP2','ZP3'],[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28], inplace=True)
kv['number'].replace(['sg','pl'],[1,2],inplace=True)
kv['preverb'].replace(['-','ა','ამო','აღ','აღმო','გა','გად','გადა','გადმო','გამო','გან','გარდ','გარდა','გარემო','გარსშემო','გარშემო','და','დამო','თანა','იავარ','მი','მიმო','მო','უკუ','შე','შემო','შთა','ჩა','ჩამო','ძალ','წა','წამო','წარ','წარმო','წინა'],[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34], inplace=True)
kv['pre2'].replace(['-'],['0'],inplace=True)
kv['root'].replace(['-'],['0'],inplace=True)
kv['sf2'].replace(['-','ავ','ამ','ე','ებ','ევ','ვ','ი','მ','ობ','ოფ'],[0,1,2,3,4,5,6,7,8,9,10],inplace=True)
kv['caus_sf'].replace(['-','ევინ','ინ'],[0,1,2],inplace=True)
kv['ending'].replace(['-'],['0'],inplace=True)
kv['sub_id'].replace('.*-','',regex=True,inplace=True)
kv['vn'].replace(['-'],['0'],inplace=True)

In [ ]:
kv[['sub_id']] = kv[['sub_id']].apply(pd.to_numeric) #to convert sub_id object type to int64
#print(kv['sub_id'])

In the UTF-8 encoding scheme, Georgian characters are represented by three bytes. The first two bytes are redundant for the conversion process. To convert Georgian text into a numeric representation, we extract the last byte from each character and sum their decimal values. For this purpose, we created the following function.

In [ ]:
def str2dec(x): #To sum last byte decimal versions for each character of a given string x
    s=0
    for i in x:
        s+=ord(i.encode('utf8')[-1:])
    return s

#kv[kv['ending'].isna()][['ending']].head(10)

Subsequently, we incorporated additional fields into the "kv" dataframe, specifically "formd", "preverbd", "pre2d", "rootd", "caus_sfd", "sf2d", and "endingd". These newly introduced fields correspond to the decimal representations of the original fields, namely "form", "preverb", "pre2", "root", "caus_sf", "sf2", and "ending". The purpose of including these fields is to store the converted decimal representations of the respective values.

In [ ]:
#To convert a Georgian string into the sum of its characters' decimal representations,
kv['formd'] = kv['form'].apply(lambda x: str2dec(x))
kv['pre2d'] = kv['pre2'].apply(lambda x: str2dec(x))
kv['rootd'] = kv['root'].apply(lambda x: str2dec(x))
kv['endingd'] = kv['ending'].apply(lambda x: str2dec(x))
kv['vnd'] = kv['vn'].apply(lambda x: str2dec(x))

Let's encode the field "vn2d" by enumerating its values from 1 to N, and store this encoding information in a dictionary called "index_vn2."

In [ ]:
index_vn = {}
vn_new_list = []

for i in kv['vn']:
    if i not in index_vn:
        index_vn[i] = len(index_vn)
    vn_new_list.append(index_vn[i])

kv['vnd']=vn_new_list

#print(index_vn)
filename = f"output_vn_index_{timestamp}.txt"
with open(filename,'w') as data:
    data.write(str(index_vn))

To facilitate further investigation, we can save the corresponding text and numeric representations for the "forms" and "verbal nouns" separately in individual files. This separation will allow for easier analysis and examination of the data.

In [ ]:
filename = f"output_form_formd_{timestamp}.csv"
kv[['form','formd']].to_csv(filename,sep=',')
filename = f"output_vn_vnd_{timestamp}.csv"
kv[['vn','vnd']].to_csv(filename,sep=',')

In [ ]:
kv.describe()

In [ ]:
kv.info()

# Setting up the Data
* Create a new dataframe called "kn_n" where we will retain only the number representation of the data.
* Simplify the dataframe by renaming the "tense_in_paradigm" column to "tense" for the sake of simplicity and clarity.

In [ ]:
kv_n = kv.loc[:,['formd','tense_in_paradigm','person','number','preverb','pre2d','rootd','sf2','caus_sf','endingd','tsch_class','morph_type','sub_id','id','vnd']]
kv_n.rename(columns={'tense_in_paradigm':'tense'}, inplace=True) # just rename 'tense_in_paradigm' with 'tense' for simplicity

To free up Memory, you can delete the "kv" dataframe.

In [ ]:
del kv

In [ ]:
kv_n.describe()

In [ ]:
kv_n.info()

To reduce the memory usage of a variable, we consider changing its data type to a less memory-intensive alternative.

In [ ]:
print(kv_n.dtypes)
print(kv_n['tense'].dtypes)

In [ ]:
kv_n[kv_n.columns[0]]=kv_n.loc[:,'formd'].astype('int16')
kv_n[kv_n.columns[1]]=kv_n.loc[:,'tense'].astype('byte')
kv_n[kv_n.columns[2]]=kv_n.loc[:,'person'].astype('byte')
kv_n[kv_n.columns[3]]=kv_n.loc[:,'number'].astype('byte')
kv_n[kv_n.columns[4]]=kv_n.loc[:,'preverb'].astype('byte')
kv_n[kv_n.columns[5]]=kv_n.loc[:,'pre2d'].astype('int16')
kv_n[kv_n.columns[6]]=kv_n.loc[:,'rootd'].astype('int16')
kv_n[kv_n.columns[7]]=kv_n.loc[:,'sf2'].astype('byte')
kv_n[kv_n.columns[8]]=kv_n.loc[:,'caus_sf'].astype('byte')
kv_n[kv_n.columns[9]]=kv_n.loc[:,'endingd'].astype('int16')
kv_n[kv_n.columns[10]]=kv_n.loc[:,'tsch_class'].astype('byte')
kv_n[kv_n.columns[11]]=kv_n.loc[:,'morph_type'].astype('byte')
kv_n[kv_n.columns[12]]=kv_n.loc[:,'sub_id'].astype('int16')
kv_n[kv_n.columns[13]]=kv_n.loc[:,'id'].astype('int16')
kv_n[kv_n.columns[14]]=kv_n.loc[:,'vnd'].astype('int16')

We can examine the new dataframe.

In [ ]:
kv_n.info()

In [ ]:
print(kv_n)

#filename = f"output_vnd_{timestamp}.csv"
#kv_n.to_csv(filename, index=False, encoding="utf-8")

## Decision Tree Model - Solution

## Train Test Split

Now, we will proceed with the task of dividing our data into a training set and a testing set.

To accomplish this, we will use the functionality provided by the scikit-learn library. This allows us to easily split our data into two distinct sets: one for training our model and the other for evaluating its performance.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = kv_n.drop('vnd',axis=1)
y = kv_n['vnd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)
#print(X_test)
#print(y)
#X_test.to_csv(r'X_test.txt', index=None, sep=';')

Import DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

Create an instance of DecisionTreeClassifier() called dtree and fit it to the training data.

In [ ]:
dtree = DecisionTreeClassifier()  #(criterion="log_loss", splitter="random", max_depth=16); 

In [ ]:
dtree.fit(X_train,y_train)

## Predictions and Evaluation of Decision Tree
Create predictions from the test set and create a classification report and a confusion matrix.

In [ ]:
predictions = dtree.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
print(classification_report(y_test,predictions))

To store the clssificaion report in the file "report.txt"

In [ ]:
report = classification_report(y_test,predictions)
report_path = "report.txt"
filename = f"output_report_{timestamp}.txt"
text_file = open(filename,"w")
n = text_file.write(report)
text_file.close()

In [ ]:
print(confusion_matrix(y_test,predictions))
cm = confusion_matrix(y_test,predictions)

# Prepare test data from a file

To prepare the test data from a file, we have the "data_vn-.csv" file that includes all the fields except for the Verbal Noun. It is particularly intriguing to observe how our trained model performs in predicting the missing Verbal Nouns.

The file "data_vn-.csv" consists of 599813 lines and will serve as our test dataset for evaluating the model's ability to predict the missing Verbal Nouns.

## Get the Data
Read in the "data_vn-.csv" file and set it to a data frame called kv.

In [ ]:
X_test2 = pd.read_csv('data_vn-', sep=';')

Check the head of the kv dataframe.

In [ ]:
#print(X_test2)
Solution = X_test2.copy()

In [ ]:
X_test2.info()

To prepare the data, we will proceed with converting the textual information into a numeric representation. This conversion is necessary to enable the utilization of machine learning algorithms that operate on numerical data.

In [ ]:
X_test2['tense_in_paradigm'].replace(['present','imperfect','conj-present','future','conditional','conj-future','aorist','optative','perfect','pluperfect','conj-perfect'],[1,2,3,4,5,6,7,8,9,10,11], inplace=True)
X_test2['morph_type'].replace(['-','active','causative','passive','stative-passive'],[0,1,2,3,4], inplace=True)
X_test2['tsch_class'].replace(['IV1','IV2','IV3','IV4','KT','MV','P1','P2','P3','RM1','RM2','RM3','RM4','RP1','RP2','RP3','RP4','RP5','RP6','RP7','T1','T2','T3','T4','T5','ZP1','ZP2','ZP3'],[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28], inplace=True)
X_test2['number'].replace(['sg','pl'],[1,2],inplace=True)
X_test2['preverb'].replace(['-','ა','ამო','აღ','აღმო','გა','გად','გადა','გადმო','გამო','გან','გარდ','გარდა','გარემო','გარსშემო','გარშემო','და','დამო','თანა','იავარ','მი','მიმო','მო','უკუ','შე','შემო','შთა','ჩა','ჩამო','ძალ','წა','წამო','წარ','წარმო','წინა','გარდმო','ზეწამო','იძულებულ','ნათელ','სრულ','უარ','უგულებელ','უვნებელ','უზრუნველ','უკვდავ','უჩინარ','ღაღად','შეურაცხ','ცხად','ხელ','წინააღ'],[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50], inplace=True)
X_test2['pre2'].replace(['-'],['0'],inplace=True)
X_test2['root'].replace(['-'],['0'],inplace=True)
X_test2['sf2'].replace(['-','ავ','ამ','ე','ებ','ევ','ვ','ი','მ','ობ','ოფ'],[0,1,2,3,4,5,6,7,8,9,10],inplace=True)
X_test2['caus_sf'].replace(['-','ევინ','ინ'],[0,1,2],inplace=True)
X_test2['ending'].replace(['-'],['0'],inplace=True)
X_test2['sub_id'].replace('.*-','',regex=True,inplace=True)

X_test2[['sub_id']] = X_test2[['sub_id']].apply(pd.to_numeric)

In [ ]:
X_test2['formd'] = X_test2['form'].apply(lambda x: str2dec(x))
X_test2['pre2d'] = X_test2['pre2'].apply(lambda x: str2dec(x))
X_test2['rootd'] = X_test2['root'].apply(lambda x: str2dec(x))
X_test2['endingd'] = X_test2['ending'].apply(lambda x: str2dec(x))

In [ ]:
X_test2.info()

In [ ]:
X_test2 = X_test2.loc[:,['formd','tense_in_paradigm','person','number','preverb','pre2d','rootd','sf2','caus_sf','endingd','tsch_class','morph_type','sub_id','id']]
X_test2.rename(columns={'tense_in_paradigm':'tense'}, inplace=True) # just rename 'tense_in_paradigm' with 'tense' for simplicity

In [ ]:
X_test2[X_test2.columns[0]]=X_test2.loc[:,'formd'].astype('int16')
X_test2[X_test2.columns[1]]=X_test2.loc[:,'tense'].astype('byte')
X_test2[X_test2.columns[2]]=X_test2.loc[:,'person'].astype('byte')
X_test2[X_test2.columns[3]]=X_test2.loc[:,'number'].astype('byte')
X_test2[X_test2.columns[4]]=X_test2.loc[:,'preverb'].astype('byte')
X_test2[X_test2.columns[5]]=X_test2.loc[:,'pre2d'].astype('int16')
X_test2[X_test2.columns[6]]=X_test2.loc[:,'rootd'].astype('int16')
X_test2[X_test2.columns[7]]=X_test2.loc[:,'sf2'].astype('byte')
X_test2[X_test2.columns[8]]=X_test2.loc[:,'caus_sf'].astype('byte')
X_test2[X_test2.columns[9]]=X_test2.loc[:,'endingd'].astype('int16')
X_test2[X_test2.columns[10]]=X_test2.loc[:,'tsch_class'].astype('byte')
X_test2[X_test2.columns[11]]=X_test2.loc[:,'morph_type'].astype('byte')
X_test2[X_test2.columns[12]]=X_test2.loc[:,'sub_id'].astype('int16')
X_test2[X_test2.columns[13]]=X_test2.loc[:,'id'].astype('int16')

In [ ]:
X_test2.info()

In [ ]:
#print(X_test2)

## Predictions and Evaluation of Decision Tree
Generate predictions from the test set and then create a classification report and a confusion matrix to evaluate the performance of the model.

In [ ]:
predictions2 = dtree.predict(X_test2)

In [ ]:
print(predictions2)

filename = f"output_predictions2_{timestamp}.txt"

with open(filename, 'w') as f:
    for line in predictions2:
        f.write(f"{line}\n")

After generating predictions from the test set, we will proceed to decode the numeric predictions back into their original Georgian text representations. This step allows us to interpret and analyze the model's outputs in a more understandable and meaningful manner. By converting the numeric predictions back to Georgian text, we can gain insights into the predicted outcomes and assess the model's performance in a linguistically meaningful context.

In [ ]:
tmppd = pd.DataFrame({'vn': predictions2})
#print(tmppd)
#To reverse index_vn2 - swap keys:values.
index_vn_rev = {i: j for j, i in index_vn.items()}
#print(index_vn)
Solution['vn'] = tmppd.replace(index_vn_rev)
#print(Solution)

In [ ]:
filename = f"output_solution{timestamp}.csv"
Solution.to_csv(filename, index=False, sep=';')

# Conclusion

The decision tree model has demonstrated excellent predictive capabilities. We conducted multiple runs of the model on the same dataset, each time using a different random_state parameter for the training and testing data split. Remarkably, in all runs, the model consistently produced identical results.

The model provides predictions with 98-99% as detailed in the **output_report.txt** file. These probabilities offer insights into the model's confidence levels for each prediction.

The corresponding outcomes can be accessed in the **output_solution.csv** file, which presents the predicted results derived from the decision tree model.